# Quantum compiler with LLM and SAT

- Compiler that lets you specify constraints in natural language using an LLM and prototype quickly
- Describe gate shapes in plain English
- LLM turns that into JSON rules
- Rules plus hardware topology go into the subgraph matcher (produces a spec of where each gate type can go)
- SAT compiler maps your circuit onto the hardware and inserts SWAPs where needed

![Compiler flow: Phase 1 (Natural language -> LLM -> JSON rules), Phase 2 (Hardware + rules -> Subgraph matcher -> Target spec), Phase 3 (Circuit + spec -> SAT compiler -> Mapped circuit)](images/compiler-flow-diagram.png)

## Set the project root path

In [ ]:
ROOT = "/home/ubuntu/LLM-integrated-Quantum-compiler"

## Imports for Hugging Face, transformers, and JSON

In [2]:
import os
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
from dotenv import load_dotenv
from huggingface_hub import login
from transformers import AutoModelForCausalLM, AutoTokenizer
from tqdm import tqdm
import sys
import json


## Load the fine-tuned LLM

In [ ]:
# load_dotenv()
# login(token=os.environ.get("HF_TOKEN"))

modelId = "rvinay73/gate-shape-compiler"
local_model_path = os.path.join(ROOT, "asplos-2026", "local_model")  # or any fixed path

config_path = os.path.join(local_model_path, "config.json")

# Check if the model is already downloaded locally
if os.path.isfile(config_path):
    print(f"Loading model and tokenizer from local path: {local_model_path}")
    tokenizer = AutoTokenizer.from_pretrained(local_model_path)
    model = AutoModelForCausalLM.from_pretrained(local_model_path, device_map="cuda", torch_dtype="auto").eval()
    print(f"Model loaded from local path")
else:
    print(f"Downloading model and tokenizer from Hugging Face and saving to {local_model_path}")
    tokenizer = AutoTokenizer.from_pretrained(modelId, use_fast=True)
    model = AutoModelForCausalLM.from_pretrained(modelId, device_map="auto", torch_dtype="auto").eval()
    
    os.makedirs(local_model_path, exist_ok=True)
    tokenizer.save_pretrained(local_model_path)
    model.save_pretrained(local_model_path)
    print("Model and tokenizer saved locally.")



Loading model and tokenizer from local path: /home/ubuntu/LLM-SAT-compiler/asplos-2026/local_model
Model loaded from local path


## Phase 1: Natural language constraints

- Describe how multi-qubit gates should be laid out
- Example: 3-qubit gates in a line, 4-qubit gates in a square
- LLM turns this into JSON rules with `nQubits`, `shape`, and `edges` (connectivity pattern between qubits)

In [10]:
sys.path.insert(0, os.path.join(ROOT, "scripts"))

from prompt_utils import build_prompt

userInput = "My 4 qubit gates should be in a square; My 3 qubit gates should be in a line"
prompt = build_prompt(userInput)

messages = [{"role": "user", "content": prompt}]

- Run the model
- Parse the JSON output

In [11]:

enc = tokenizer.apply_chat_template(
    conversation=messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt",
)


enc = enc.to(model.device)


out = model.generate(**enc, max_new_tokens=80)


promptLen = enc["input_ids"].size(-1)

response = tokenizer.decode(out[0][promptLen:], skip_special_tokens=True)


s = response
try:
    parsed = json.loads(s)
    print(json.dumps(parsed, indent=3))
except json.JSONDecodeError:
    print(s) 

{
   "rules": [
      {
         "nQubits": 3,
         "shape": "line",
         "edges": [
            [
               0,
               1
            ],
            [
               1,
               2
            ]
         ]
      },
      {
         "nQubits": 4,
         "shape": "square",
         "edges": [
            [
               0,
               1
            ],
            [
               0,
               3
            ],
            [
               1,
               2
            ],
            [
               2,
               3
            ]
         ]
      }
   ]
}


## Phase 2: Subgraph matcher

- Takes LLM rules and hardware graph (raw edges)
- Finds all places where each gate shape can fit on the hardware
- Writes a target spec
- Spec lists which physical qubits each gate type can use

In [6]:
sys.path.insert(0, os.path.join(ROOT, "scripts"))
from build_arch_from_llm_rules import load_raw_hardware, load_llm_rules, build_target_spec


llm_data = json.loads(response)
llm_rules = load_llm_rules(llm_data)

# Raw hardware: path to edges-only JSON
raw_path = os.path.join(ROOT, "asplos-2026", "sample-input-files", "tokyo-edges.json")
raw_spec = load_raw_hardware(raw_path)

target_spec = build_target_spec(raw_spec, llm_rules)

# Save target hardware spec to a JSON file
out_path = os.path.join(ROOT, "asplos-2026", "sample-output-files",  "arch_from_llm.json")
with open(out_path, "w") as f:
    json.dump(target_spec, f, indent=2)
print("Wrote", out_path)

Wrote /home/ubuntu/LLM-SAT-compiler/asplos-2026/sample-output-files/arch_from_llm.json


## Phase 3: SAT compiler

- Maps logical qubits to physical qubits
- Inserts SWAPs so every gate can run
- Takes circuit and hardware spec, outputs mapped circuit

**SAT formulation for multi qubit gates**

For each n-qubit gate at step k with logical qubits q₁,…,qₙ, SubgraphMatches = allowed ordered n-tuples from the hardware spec. The constraint:
```
⋁_{M_j ∈ SubgraphMatches}  (  map(q₁,pⱼ₁,k) ∧ map(q₂,pⱼ₂,k) ∧ … ∧ map(qₙ,pⱼₙ,k)  )
```

Meaning: at least one tuple M_j must hold every logical qubit q_i , which maps to the corresponding physical qubit p_ji at step k.

In [7]:
sys.path.insert(0, os.path.join(ROOT, "src"))

import satmap
import hardware_spec as hw_spec

prog_path = os.path.join(ROOT, "asplos-2026", "sample-input-files", "sample-circuit.qasm")
hardware_spec_path = os.path.join(ROOT, "asplos-2026", "sample-output-files", "arch_from_llm.json")
out_path = os.path.join(ROOT, "asplos-2026", "sample-output-files", "output-circuit.qasm")
AUX_DIR = os.path.join(ROOT, "asplos-2026", "aux_files")

spec = hw_spec.load_spec(hardware_spec_path)
cm = hw_spec.build_cm_from_spec(spec)

os.makedirs(AUX_DIR, exist_ok=True)
base = os.path.splitext(os.path.basename(prog_path))[0]

os.chdir(ROOT)

stats, qasm = satmap.transpile(
    prog_path,
    cm,
    swapNum=1,
    cnfname=os.path.join(AUX_DIR, f"prob_{base}"),
    sname=os.path.join(AUX_DIR, f"sol_{base}"),
    slice_size=50,
    max_sat_time=28800,
    routing=True,
    bounded_above=True,
    hardware_spec=hardware_spec_path,
)

with open(out_path, "w") as f:
    f.write(qasm)
print("Wrote", out_path)

Wrote /home/ubuntu/LLM-SAT-compiler/asplos-2026/sample-output-files/output-circuit.qasm


### TODO

- Add type check functions for LLM output verification
- Add circuit generator with sample gates (with multi qubit gates)
- Add hardware map generator
- Draw hardware map
- Add a function to highlight where each type of gate can run
- Throw formatted errors if constraints cannot be met in the hardware map
- Add functions for drawing input and output circuit